# Benchmark: new vs. old vector-classification pipeline

Scores the **new** `rastervec` Vector-Classification + OCR pipeline against the
**old** archive/legacy pipeline on the same ground truth, using the independent
metric suite in `Evaluation/Evaluate/metrics.py` (see `EVAL_METRICS.md` for every
metric's formula). Each metric is a `Ratio(numerator, denominator)`: per-page
reports show absolute counts, aggregates are **micro-averaged** (sum numerators /
sum denominators, not the mean of per-page ratios).

- **Current (new)**: `rastervec`'s own `Pipeline.STAGES` chain (layer/color
  separation -> 12-step Vector_Classification -> FAST text detect -> PaddleOCR),
  via `pipeline.run_page_context`.
- **Archive (legacy / old)**: `archive/raster_parser/main_pipeline_extract.extract`,
  run completely unmodified via `Evaluation/Evaluate/legacy_adapter.py`.

Each `(pdf, page)` is a `PageTask` run by `rastervec.Reader.Parallel` — its own
ground truth, conversion, pipeline run, evaluation, and per-page PDF output, all
in one process. `BENCH_WORKERS > 1` fans the pages across a spawn process pool
(model caches warmed once up front, so the first run is safe too). Legacy
per-page parallelism is the only useful lever there — archive's own `workers`
arg never reaches the Type-2 native→OCR path the benchmark runs.

### Dataset

`collect_dataset(DATASET_ROOT)` recursively walks one directory tree for both
`.pdf` files and label-sidecar `.json` files and pairs them into a flat
`(pdf, page)` dataset, keyed by real filesystem path — a label file that names a
root-folder PDF is the *same* item as its discovered copy, so no page runs
twice. A tree may mix labelled and unlabelled PDFs freely.

### Ground truth — two sources, scored separately

- **auto** — `auto_label_pdf`, derived from the PDF's own native text,
  independent of either pipeline. Always generated for every benchmarked page.
- **manual** — human-entered `LabelEntry`s from a `manual_label.py` sidecar
  `.json` discovered in the tree, attached to the pages they name. Conversion
  preserves every pre-existing vector path byte-for-byte, so a manually-labelled
  vector cluster still lines up in the converted PDF.

`split_labelset_by_source` keeps the two apart: every pipeline is scored once
against auto labels and once against manual labels, so `current/auto`,
`current/manual`, `legacy/auto`, `legacy/manual` each get their own aggregate
and chart series.

### Output

- **Per-page** evaluation reports and per-page stage timing → `RESULTS_TXT`
  (current) / `<stem>_legacy.txt` (legacy). Not printed.
- **Printed**: only the aggregated accuracy metrics, the aggregated per-stage +
  per-page timing distribution (min / Q1 / median / mean / Q3 / max), and the
  charts.
- **Per-page PDFs** (when `RECONSTRUCT_DIR` is set): `<stem>_p<N>_groundtruth.pdf`
  (label text), `_<pipeline>.pdf` (text reconstruction, text-only — no drawing
  layer), `_<pipeline>_input.pdf` (the exact PDF fed to that pipeline), and
  `_<pipeline>_boxes_<auto|manual>.pdf` (green = matched gt/pred, yellow =
  predicted box over no gt, red = gt no prediction reached).

This is a **sanity/regression check**, not a real A/B: the new pipeline is
expected to score at least as well as the old one.


In [ ]:
import io
import math
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "benchmark_vector_classification.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from rastervec.Evaluation.Evaluate.benchmark import (
    aggregate_results,
    distribution_stats,
    format_aggregate,
    format_timing_report,
    summarize_stage_timings,
)
from rastervec.Evaluation.Evaluate.metrics import LOWER_IS_BETTER, METRIC_GROUPS, MetricConfig
from rastervec.logging_setup import configure_logging
from rastervec.pipeline import Pipeline
from rastervec.Reader.dataset import collect_dataset
from rastervec.Reader.Parallel import PageTask, default_worker_count, run_benchmark

configure_logging()


## Parameters

`DATASET_ROOT` is a directory tree searched **recursively** for both `.pdf`
files and label-sidecar `.json` files (the `manual_label.py` / `auto_label.py`
`LabelSet` format). `collect_dataset` pairs them into one flat `(pdf, page)`
dataset (keyed by real path, so a label file that names a root-folder PDF is the
same item as its discovered copy — one run per page):

- A PDF a label file references → benchmarked on exactly the pages that file
  names, carrying its `source="manual"` entries.
- A PDF with no label file → benchmarked on its first `PAGES_PER_PDF` pages,
  auto labels only.

Either way, **auto** ground truth (`auto_label_pdf`, from the PDF's own native
text) is generated per page, and `split_labelset_by_source` scores auto and
manual separately (`current/auto`, `current/manual`, `legacy/auto`,
`legacy/manual`).

- `RESULTS_TXT` — per-page evaluation reports + per-page stage timing are
  written here (current pipeline) / to `<stem>_legacy.txt` (legacy). Only the
  aggregated metrics and the aggregated timing distribution are printed.
- `RECONSTRUCT_DIR` — per-page reconstruction / input / box-overlay PDFs
  (`None` to skip).
- `BENCH_WORKERS` — process-pool size for the per-page loop (1 = serial). Each
  worker holds its own PaddleOCR + torch model; `run_benchmark` warms the model
  caches once up front so `> 1` is safe on the first run. `default_worker_count()`
  is a conservative suggestion.

Both pipelines run real PaddleOCR, and the archive pipeline also shells out to
LibreOffice, so keep `PAGES_PER_PDF` small.


In [ ]:
# Directory tree searched recursively for .pdf + label .json files.
DATASET_ROOT = PROJECT_ROOT / "references"

# Page cap for PDFs in the tree that have no sidecar label file (auto labels
# only). PDFs a label file references use exactly the pages it names.
PAGES_PER_PDF = 3

# Per-page evaluation reports + per-page stage timing go here (current) /
# RESULTS_TXT.with_name(stem + "_legacy.txt") (legacy). Only aggregates print.
RESULTS_TXT = PROJECT_ROOT / "benchmark_results.txt"

# Per-page PDFs are written here (None disables): <stem>_p<N>_groundtruth.pdf,
# _<pipeline>.pdf (text reconstruction), _<pipeline>_input.pdf (the exact PDF
# fed to that pipeline), _<pipeline>_boxes_<auto|manual>.pdf (green = matched
# gt/pred, yellow = predicted box over no gt, red = gt no prediction reached).
RECONSTRUCT_DIR = PROJECT_ROOT / "benchmark_reconstructions"

# Process-pool size for the per-page benchmark loop (1 = serial). Each worker
# loads its own PaddleOCR + torch model, so memory is the limit -- see
# default_worker_count(). run_benchmark() warms the model caches in-process
# before spawning, so BENCH_WORKERS > 1 is safe on the first run too.
BENCH_WORKERS = 1

# How many PaddleOCR cluster renders each page returns for the showcase grid
# (~50/50 OCR passes vs blank failures), and the cap on how many to plot.
SHOWCASE_PER_PAGE = 4
SHOWCASE_N = 20
SHOWCASE_SEED = 0

# MetricConfig.iou_edge_min -- minimum IoU for a gt<->prediction localisation
# edge (the N:1 fallback assignment + miss-attribution group match).
IOU_EDGE_MIN = MetricConfig().iou_edge_min

# Archive's raster-fallback stage shells out to LibreOffice (`soffice`) to
# strip native content before its Type-4 OCR/autotrace pass -- set True only
# if LibreOffice is installed and on PATH; otherwise this stays Type-2-only
# (native + fill-vector OCR), which is still a fair comparison since the
# current pipeline being benchmarked has no raster-image stage either.
ENABLE_ARCHIVE_RASTER_PASS = False


In [ ]:
# Recursively collect every PDF + every label file under DATASET_ROOT into one
# flat (pdf, page) dataset (see rastervec.Reader.dataset.collect_dataset).
dataset = collect_dataset(DATASET_ROOT, pages_per_pdf=PAGES_PER_PDF)

pdf_pages = [(d.pdf_path, d.page_index) for d in dataset]
# manual_entries[(pdf_path, page_index)] -> list[LabelEntry] (source == "manual")
manual_entries = {
    (d.pdf_path, d.page_index): list(d.manual_entries) for d in dataset
}

n_manual = sum(len(v) for v in manual_entries.values())
n_labelled_pages = sum(1 for v in manual_entries.values() if v)
print(f"{len(pdf_pages)} (pdf, page) pair(s) under {DATASET_ROOT}; "
      f"{n_manual} manual label(s) across {n_labelled_pages} page(s)")

In [ ]:
def build_tasks(pipeline: str) -> list[PageTask]:
    """One PageTask per (pdf, page) in the dataset (already de-duplicated by
    collect_dataset, so no page runs twice). Each task does its own ground
    truth (auto + any manual), conversion, pipeline run, evaluation, and
    per-page PDF output -- see rastervec.Reader.Parallel.benchmark_jobs."""
    return [
        PageTask(
            pdf_path=pdf_path,
            page_index=page_index,
            manual_entries=list(manual_entries.get((pdf_path, page_index), [])),
            iou_edge_min=IOU_EDGE_MIN,
            pipeline=pipeline,
            reconstruct_dir=str(RECONSTRUCT_DIR) if RECONSTRUCT_DIR else None,
            showcase_per_page=SHOWCASE_PER_PAGE if pipeline == "current" else 0,
            enable_archive_raster_pass=ENABLE_ARCHIVE_RASTER_PASS,
            showcase_seed=SHOWCASE_SEED,
        )
        for pdf_path, page_index in pdf_pages
    ]


def collect_results(results: list, txt_path: Path) -> tuple[list, list, list, list]:
    """Split a run's PageResults into (auto, manual, stage_timings,
    showcase) and write every per-page report block + failure to txt_path."""
    auto, manual, timings, showcase, lines = [], [], [], [], []
    for r in results:
        if r.error is not None:
            lines.append(f"[{r.pipeline}] {r.pdf_path} page {r.page_index} failed: {r.error}")
            continue
        lines.extend(r.report_blocks)
        lines.append(
            f"  stage timing: total={r.total_seconds:.2f}s"
            + (f"  ({', '.join(f'{k}={v:.2f}s' for k, v in r.stage_durations.items())})"
               if r.stage_durations else "")
        )
        if r.auto is not None:
            auto.append(r.auto)
        if r.manual is not None:
            manual.append(r.manual)
        if r.stage_durations:
            timings.append(r.stage_durations)
        showcase.extend(r.showcase)
    txt_path.write_text("\n\n".join(lines) + "\n", encoding="utf-8")
    return auto, manual, timings, showcase


## Run the pipelines

The current (new) and legacy (old) runs are separate cells so you can run
either on its own. Each produces two result lists â€” `*_auto_results` and
`*_manual_results` (the latter stays empty unless a sidecar `.json` supplied
manual labels for a page). The aggregate + chart cells below tolerate any of
the four lists being empty. Per-page reports are written to the results `.txt`
files, not printed.

### Current (new) pipeline

In [ ]:
current_results = run_benchmark(
    build_tasks("current"), workers=BENCH_WORKERS, desc="current",
)
(
    current_auto_results,
    current_manual_results,
    current_stage_timings,
    current_showcase,
) = collect_results(current_results, RESULTS_TXT)

print(f"per-page reports -> {RESULTS_TXT}  "
      f"({len(current_auto_results)} auto, {len(current_manual_results)} manual page-scores; "
      f"{sum(1 for r in current_results if r.error)} failed)")


### PaddleOCR render showcase (current pipeline)

Up to `SHOWCASE_PER_PAGE` of the cluster images each page actually fed to
PaddleOCR (rendered exactly as `RenderOCR.ocr_cluster` does, returned as PNG
bytes on the `PageResult`), pooled across every page and sampled with
`SHOWCASE_SEED`, split ~50/50 between **PASS** (non-blank OCR reading) and
**FAIL** (blank). If one side is short, the rest is topped up from the pool.
Needs the current-pipeline cell above to have run.


In [ ]:
pool = globals().get("current_showcase", [])
passed = [s for s in pool if s.passed]
failed = [s for s in pool if not s.passed]
print(f"pool: {len(pool)} cluster renders  ({len(passed)} passed, {len(failed)} blank)")

rng = random.Random(SHOWCASE_SEED)
half = SHOWCASE_N // 2
pick = rng.sample(passed, min(half, len(passed)))
pick += rng.sample(failed, min(SHOWCASE_N - len(pick), len(failed)))
chosen = {id(s) for s in pick}
rest = [s for s in pool if id(s) not in chosen]
rng.shuffle(rest)
pick += rest[: max(0, SHOWCASE_N - len(pick))]
rng.shuffle(pick)

if not pick:
    print("nothing to showcase -- run the current-pipeline cell first")
else:
    cols = 4
    rows = math.ceil(len(pick) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.0), squeeze=False)
    for ax, sample in zip(axes.flat, pick):
        ax.imshow(Image.open(io.BytesIO(sample.png)))
        label = f'PASS  "{sample.text}"' if sample.passed else "FAIL  (blank OCR)"
        ax.set_title(label[:46], color="#1a7f37" if sample.passed else "#cf222e", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    for ax in axes.flat[len(pick):]:
        ax.axis("off")
    n_pass = sum(1 for s in pick if s.passed)
    fig.suptitle(
        f"Current-pipeline PaddleOCR cluster renders "
        f"({n_pass} pass / {len(pick) - n_pass} fail)",
        y=1.0,
    )
    plt.tight_layout()
    plt.show()


### Legacy (old / archive) pipeline

Independent of the current-pipeline cell above — run it only if you have the
archive tree at `<repo_root>/archive/` (it is not checked into this repo).
Without it, every `legacy` `PageTask` captures a `ModuleNotFoundError:
raster_parser` into `PageResult.error`, the loop finishes, and both legacy
result lists stay empty. Legacy runs on the **original** native PDF (no
Conversion — archive reads native text directly), so its
`_legacy_input.pdf` is that original page and its miss-attribution metrics
come back `n/a` (no `clustering` / `fast_dropped` / `ocr_failed`).


In [ ]:
_legacy_txt = RESULTS_TXT.with_name(RESULTS_TXT.stem + "_legacy.txt")

legacy_results = run_benchmark(
    build_tasks("legacy"), workers=BENCH_WORKERS, desc="legacy",
)
legacy_auto_results, legacy_manual_results, _, _ = collect_results(legacy_results, _legacy_txt)
legacy_page_totals = [r.total_seconds for r in legacy_results if r.error is None]

print(f"per-page legacy reports -> {_legacy_txt}  "
      f"({len(legacy_auto_results)} auto, {len(legacy_manual_results)} manual page-scores; "
      f"{sum(1 for r in legacy_results if r.error)} failed)")


In [ ]:
def print_aggregate(name, results):
    agg = aggregate_results(results)
    print(format_aggregate(agg, len(results), label=name))
    print()
    return agg


current_auto_agg = print_aggregate("current / auto", globals().get("current_auto_results", []))
current_manual_agg = print_aggregate("current / manual", globals().get("current_manual_results", []))
legacy_auto_agg = print_aggregate("legacy / auto", globals().get("legacy_auto_results", []))
legacy_manual_agg = print_aggregate("legacy / manual", globals().get("legacy_manual_results", []))

# --- timing distribution (min / Q1 / median / mean / Q3 / max) ---
timing_summary = summarize_stage_timings(
    globals().get("current_stage_timings", []), Pipeline.stage_keys(),
)
timing_report = format_timing_report(
    timing_summary, title="Current pipeline -- per-stage timing (seconds)",
)
print(timing_report)

legacy_totals = globals().get("legacy_page_totals", [])
legacy_timing_line = ""
if legacy_totals:
    lt = distribution_stats(legacy_totals)
    legacy_timing_line = (
        "Legacy pipeline -- per-page total (seconds): "
        f"n={lt['n']}  min={lt['min']:.2f}  q1={lt['q1']:.2f}  "
        f"median={lt['median']:.2f}  mean={lt['mean']:.2f}  "
        f"q3={lt['q3']:.2f}  max={lt['max']:.2f}"
    )
    print()
    print(legacy_timing_line)

# Append the timing tables to the current-pipeline results .txt file.
if globals().get("current_stage_timings") and RESULTS_TXT.exists():
    with RESULTS_TXT.open("a", encoding="utf-8") as fh:
        block = timing_report
        if legacy_timing_line:
            block += "\n" + legacy_timing_line
        fh.write("\n\n" + block + "\n")


## Reading the results

- The **new** pipeline (`current/*`) is expected to match or beat the **old**
  archive pipeline (`legacy/*`) on every metric. A page where the old pipeline
  is ahead is a concrete regression — open that page's block in `RESULTS_TXT`
  and check its `per_stage_miss_counts` for where the new pipeline lost the text
  (`gt_miss_attributed_to_*` breaks the same funnel out as fractions).
- **auto vs. manual**: auto ground truth is every native-text line; manual
  ground truth is only the clusters a human labelled in `manual_label.py`.
  Manual is usually the stricter, more curated set — a big gap between
  `*/auto` and `*/manual` on the same pipeline points at auto labels that don't
  correspond to a real recoverable text cluster (or vice-versa).
- **Timing**: the aggregate cell prints the per-stage and per-page-total
  distribution (min / Q1 / median / mean / Q3 / max); per-page stage timing is
  in `RESULTS_TXT`. `ocr_compare` and `fast_text_detect` dominate — a jump in
  either stage's max vs. median points at a pathological page.
- **`_<pipeline>_boxes_*.pdf`** in `RECONSTRUCT_DIR`: green boxes are matched
  gt/pred, red boxes are ground truth no prediction reached, yellow boxes are
  predictions sitting over no ground truth. `_groundtruth.pdf` next to
  `_<pipeline>.pdf` (same page size) shows whether the found text lands where the
  ground-truth text is; `_<pipeline>_input.pdf` is the exact PDF that pipeline
  ran on.
- **PaddleOCR render showcase**: the FAIL tiles are the clusters that reached
  OCR but came back blank (folded into `drawing_vectors`) — scan them for
  renders that clearly *are* text the OCR should have read (a rendering bug, a
  bad crop, an under-/over-merged cluster) vs. genuine non-text.


In [ ]:
import math

import matplotlib.pyplot as plt

_series = [
    (name, agg)
    for name, agg in [
        ("current/auto", globals().get("current_auto_agg")),
        ("current/manual", globals().get("current_manual_agg")),
        ("legacy/auto", globals().get("legacy_auto_agg")),
        ("legacy/manual", globals().get("legacy_manual_agg")),
    ]
    if agg is not None
]

_DOWN = " " + chr(0x2193)  # down arrow = "lower is better"
_groups = list(METRIC_GROUPS)
fig, axes = plt.subplots(len(_groups), 1, figsize=(11, 3.1 * len(_groups)))
if len(_groups) == 1:
    axes = [axes]

for ax, (dimension, names) in zip(axes, _groups):
    x = range(len(names))
    n = max(len(_series), 1)
    width = 0.8 / n
    for i, (sname, agg) in enumerate(_series):
        offset = (i - (n - 1) / 2) * width
        vals = []
        for m in names:
            v = agg.get(m)
            vals.append(0.0 if (v is None or math.isnan(v)) else v)
        ax.bar([j + offset for j in x], vals, width, label=sname)
    ax.set_xticks(list(x))
    ax.set_xticklabels(
        [m + (_DOWN if m in LOWER_IS_BETTER else "") for m in names],
        rotation=20, ha="right", fontsize=7,
    )
    ax.set_ylim(0, 1)
    ax.set_title(dimension)
    ax.legend(fontsize=7, loc="lower right")

fig.suptitle(
    "Metric suite -- current vs legacy, auto vs manual ground truth "
    "(down arrow = lower is better; bars at 0 may be n/a)"
)
plt.tight_layout()
plt.show()


## Files written

- `RESULTS_TXT` — per-page `format_report` blocks + per-page stage timing for
  the current pipeline, with the aggregated timing tables appended at the end.
- `RESULTS_TXT.with_name(<stem>_legacy.txt)` — the same for the legacy pipeline
  (per-page total time only).
- `RECONSTRUCT_DIR/<stem>_p<N>_...pdf` — per page:
  - `_groundtruth.pdf` — the label text, redrawn.
  - `_<pipeline>.pdf` — the pipeline's found text, redrawn (text-only, no
    drawing layer).
  - `_<pipeline>_input.pdf` — the exact PDF that pipeline was run on (converted
    vector-text for `current`, the original page for `legacy`).
  - `_<pipeline>_boxes_<auto|manual>.pdf` — pred-vs-ground-truth boxes: green =
    a gt/pred that overlaps, yellow = a predicted box over no gt, red = a gt no
    prediction reached.


In [ ]:
import matplotlib.pyplot as plt

# Per-stage median wall-clock (current pipeline), with min-max whiskers.
_summary = globals().get("timing_summary", {})
_stages = [k for k in _summary if k != "total"]
if not _stages:
    print("no timing data -- run the current-pipeline cell first")
else:
    medians = [_summary[k]["median"] for k in _stages]
    lo = [_summary[k]["median"] - _summary[k]["min"] for k in _stages]
    hi = [_summary[k]["max"] - _summary[k]["median"] for k in _stages]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(range(len(_stages)), medians, yerr=[lo, hi], capsize=3, color="#4c78a8")
    ax.set_xticks(range(len(_stages)))
    ax.set_xticklabels(_stages, rotation=30, ha="right")
    ax.set_ylabel("seconds")
    total = _summary["total"]
    ax.set_title(
        f"Current pipeline per-stage time (median, min-max whiskers)  "
        f"|  per-page total median {total['median']:.1f}s "
        f"(min {total['min']:.1f}s / max {total['max']:.1f}s)"
    )
    plt.tight_layout()
    plt.show()

## Reading the results

- The **new** pipeline (`current/*`) is expected to match or beat the **old**
  archive pipeline (`legacy/*`) on every metric. A page where the old pipeline
  is ahead is a concrete regression — check `per_stage_miss_counts` in that page's
  `format_report` block for where the new pipeline lost the text.
- **auto vs. manual**: auto ground truth is every native-text line; manual
  ground truth is only the clusters a human labelled in `manual_label.py`.
  Manual is usually the stricter, more curated set — a big gap between
  `*/auto` and `*/manual` on the same pipeline points at auto labels that don't
  correspond to a real recoverable text cluster (or vice-versa).
- **`_<pipeline>_boxes_*.pdf`** in `RECONSTRUCT_DIR`: green = matched gt/pred,
  red = gt no prediction reached, yellow = prediction over no gt. `_groundtruth.pdf`
  next to `_<pipeline>.pdf` (same page size) shows whether the found text lands
  where the ground-truth text is; `_<pipeline>_input.pdf` is the exact PDF that
  pipeline ran on.
